# Example 3: Flip-flop shift register and LUTRAM injection

This example does two campaigns on one design through BSCAN:

- Part 1: a shift register with a feedback loop, made of flip-flops.
- Part 2: a distributed RAM (LUTRAM).

USER1 starts and stops the clock of the shift register and of the LUTRAM. The script reads the shift register through USER2. USER3 and USER4 give access to the LUTRAM (address and data).

![Block diagram of ff_lut_top. USER1 controls a clock gate. The gated clock drives a clock divider with a shift register in one pblock, and a LUTRAM in another pblock. USER2 reads the shift register. USER3 and USER4 give the address and the data of the LUTRAM.](images/example3_ff_lut_top_abstract.svg)

## Part 1: flip-flops

Part 1 injects faults into the flip-flops inside the shift-register's pblock area (`pblock_ff_sr_inst`) (`TargetMemory.FF`). The server finds the flip-flops from the LL file. If the pblock contains no flip-flops, the script stops.

Each trial does these steps:

1. It lets the clock operate for 2 s, and then it stops the clock.
2. It reads the shift register.
3. It upsets one flip-flop.
4. It reads the shift register again and prints the two values.
5. It starts the clock again.

The script does not correct the upsets.

## Part 2: LUTRAM

Part 2 injects faults into the LUTRAM in `pblock_LUTRAM` (`TargetMemory.LUTRAM`). The script does these steps:

1. With the clock on, it writes a known pattern into each word. Then it reads back each word to make sure that the pattern is correct.
2. It stops the clock and injects one upset each 0.5 s, until you press Enter.
3. It starts the clock again and reads back the full array.
4. It prints each word with the positions of its incorrect bits.

## The calls of the example

| Call | Use in the example |
| --- | --- |
| `SkraftFpga.connect(ip=..., port=...)`, `client.attach(SERIAL_NUMBER)` | Connect to the SK-RAFT server and attach to the device, in `with` statements that release the device and close the connection. |
| `Design(bit, ll, xdc)`, `dev.uploadDesignToServer(design)`, `dev.programFPGA(design)` | Give the design files, upload them to the server and program the FPGA. |
| `dev.configure_seed(0)` | Set the seed of the random injections. The same seed gives the same upsets in each run. |
| `BscanPrimitive(user, length)`, `dev.write_bscan(primitive, value)`, `dev.read_bscan(primitive)` | Start and stop the clock (USER1), read the shift register (USER2), and select, write and read a LUTRAM word (USER3, USER4). |
| `dev.bit_count(TargetMemory.FF, pblock=PBLOCK_SR)`, `dev.bit_count(TargetMemory.LUTRAM, pblock=PBLOCK_LUTRAM)` | Get the number of flip-flop bits and LUTRAM bits of each pblock. |
| `dev.inject_random(TargetMemory.FF, pblock=PBLOCK_SR)` | Part 1: inject one upset into a random flip-flop of the shift register. |
| `dev.inject_random(TargetMemory.LUTRAM, pblock=PBLOCK_LUTRAM)` | Part 2: inject one upset into a random bit of the LUTRAM. |

| Constant | Default | Description |
| --- | --- | --- |
| `SR_LEN` | `8` | The width of the shift register. It must be equal to the `sr_len` generic of the design. |
| `LUT_ADDR_BITS` / `LUT_DEPTH` | `6` / `64` | The address width and the number of words of the LUTRAM. |
| `LUT_DATA_LEN` | `32` | The word width of the LUTRAM. It must be equal to the `lut_data_len` generic of the design. |
| `PBLOCK_SR` / `PBLOCK_LUTRAM` | `"pblock_ff_sr_inst"` / `"pblock_LUTRAM"` | The pblocks for parts 1 and 2, as the XDC file names them. |

## Result

Sample output of a run on a Basys 3 (shortened). The frame addresses, the bit counts and the error counts depend on the board, the build and the seed.

```
Example 3: Flip-flop shift register and LUTRAM injection
Programmed 210183BFD452A.
Part 1: 8 flip-flop bits inside pblock_ff_sr_inst.
Press Enter to move on.
     #  FAR       WORD  BIT  BEFORE    AFTER
     1  00400f9f    97   28  00000100  00100100
     2  00400f9f    97   28  10010000  10110000
     3  00400f9f    97    4  11000010  11000011
     4  00400f9f    97   28  00001111  00101111
Part 1: done after 4 trials.
Re-programming before Part 2.
Part 2: all 64 LUTRAM words read back correctly. Injecting random upsets, not corrected - press Enter to move on.
Part 2: 2048 LUTRAM bits inside pblock_LUTRAM.
     #  FAR       WORD  BIT
     1  00000a21     3   27
     2  00000a23     3    0
...
Part 2: 7 upsets injected.
  ADDR  CORRECT                           READ                              CORRUPTED BITS
     0  10100101101001011010010110100101  10100101101001011010010110100101  -
...
     9  10100101101001011010010110101100  10100101101001011010010110100100  3
...
    55  10100101101001011010010110010010  10101101111001011010010110010010  22, 27
...
Part 2: 6/64 words are corrupted, 7 bits in total.
Re-programming to clear the device.
Example 3 done.
```

The script runs part 1, programs the FPGA again, and runs part 2. Then it programs the FPGA one more time, to make sure that the configuration is clean.

If you stop the script during a campaign, the configuration of the device can stay incorrect. In that case, program the FPGA again before you do other work.

## Run the example

### 0: Install the module

This will restart the kernel, a popup will show up, just ignore it.

In [1]:
%pip install ../../api/python

import IPython; IPython.get_ipython().kernel.do_shutdown(restart=True)

Defaulting to user installation because normal site-packages is not writeable
Processing /home/wallmenis/git/sk-raft-client/api/python
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for skraft: filename=skraft-1.0.0-py3-none-any.whl size=45974 sha256=8766d17ca23b055e96a4dfa6dd72688436773288be5c3893690f8fa4388aa36f
  Stored in directory: /tmp/pip-ephem-wheel-cache-qyp2g_9i/wheels/cc/7e/0f/603cd9eaf8e594f1c766c921daeeef2440d17e0cbb287bfb57
Successfully built skraft
  Attempting uninstall: skraft
    Found existing installation: skraft 1.0.0
    Uninstalling skraft-1.0.0:
      Successfully uninstalled skraft-1.0.0
Note: you may need to restart the kernel to use updated packages.


{'status': 'ok', 'restart': True}

### 1: Extract the design and bitstream files

On Linux

In [1]:
%%bash
cd ../hw/design_examples/
./unzip_designs.sh

Design files extracted to /home/wallmenis/git/sk-raft-client/get_started/hw/design_examples


On Windows

In [ ]:
%%script pwsh
cd ..\hw\design_examples\
.\unzip_designs.bat

### 2: Add the imports

In [2]:
import os
import random
import threading
import time

from skraft import Design, TargetMemory, ReadbackOption, SkraftError, SkraftFpga, BscanPrimitive

SK_RAFT_SERVER_IP = "127.0.0.1"
SK_RAFT_SERVER_PORT = 50051

### 3: Start the server

In [7]:
%%script bash --bg
sk-raft

### 4: List devices
Connect the devices you want to test before running list devices.

In [3]:
def main() -> int:
    try:
        # SK-RAFT: open the connection. Then list the connected boards.
        with SkraftFpga.connect(ip=SK_RAFT_SERVER_IP,
                                port=SK_RAFT_SERVER_PORT) as client:
            devices = client.get_devices()
    except SkraftError as e:
        print(f"Cannot list the boards: {e}")
        return 1
    except KeyboardInterrupt:
        print("Interrupted.")
        return 130

    if not devices:
        print("No board is connected to the SK-RAFT server.")
        return 1
    print(f"{'SERIAL NUMBER':16s} PART NUMBER")
    for device in devices:
        print(f"{device.serial_number:16s} {device.part_number}")
    return 0


if __name__ == "__main__":
    main()

SERIAL NUMBER    PART NUMBER
210183BA2524A    XC7A35T


### 5: Setup the code
Don't forget to change the Serial Number (`SERIAL_NUMBER`) and Board (`BOARD`) to the ones you desire from the list!

In [4]:
SERIAL_NUMBER = "210183BA2524A"

# The design for both parts: a flip-flop shift register and a LUTRAM.
# The script gives all three files at the start:
#   The bitstream (.bit): to program the device.
#   The XDC file (.xdc): to find the pblocks.
#   The LL file (.ll): to find the BRAM/FF/LUTRAM bits that the design uses.
# uploadDesignToServer() sends the files to the server. The client sends
# only the file name of each path, not the directory.
#
# BOARD is the board of your target device: "basys3" or "zedboard", or
# the name of your own board (see hw/README.md). The script reads the
# design files from hw/design_examples/<BOARD>/ff_lut_top/. Extract
# design_examples.zip there, or build the design with create_project.tcl.
# The paths are relative to src/. Run the script from that folder.
BOARD = "basys3"
DESIGN_DIR = f"../hw/design_examples/{BOARD}/ff_lut_top"

design = Design(
    bit=f"{DESIGN_DIR}/ff_lut_top.bit",
    ll =f"{DESIGN_DIR}/ff_lut_top.ll",
    xdc=f"{DESIGN_DIR}/board.xdc",
)

# The two pblocks to test, with the names from the XDC file.
PBLOCK_SR = "pblock_ff_sr_inst"
PBLOCK_LUTRAM = "pblock_LUTRAM"

# The generics of the design. If you build the design again with
# different values, set these constants to the same values.
SR_LEN = 8
# 6 address bits give 64 LUTRAM words.
LUT_ADDR_BITS = 6
LUT_DEPTH = 64
LUT_DATA_LEN = 32

# BSCAN registers: (USER number, width in bits)
USER_CLK_EN = BscanPrimitive(1, 1)
USER_SR = BscanPrimitive(2, SR_LEN)
USER_LUT_ADDRESS = BscanPrimitive(3, LUT_ADDR_BITS)
USER_LUT_DATA = BscanPrimitive(4, LUT_DATA_LEN)

# Masks for the values that the script writes to and reads from BSCAN.
SR_MASK = (1 << SR_LEN) - 1
LUT_DATA_MASK = (1 << LUT_DATA_LEN) - 1

### 6: Define some helper functions

In [5]:
# Start or stop the clock of the shift register and the LUTRAM through USER1.
def enable_clock(dev, enabled: bool) -> None:
    # SK-RAFT: set the clock enable bit.
    dev.write_bscan(USER_CLK_EN, 1 if enabled else 0)


# Read the bits of the shift register from USER2.
def read_sr(dev) -> int:
    # SK-RAFT: read the shift register. Then unpack the bits.
    data = dev.read_bscan(USER_SR)
    return int.from_bytes(data, "big") & SR_MASK


# Select one LUTRAM address through USER3.
def select(dev, address: int) -> None:
    # SK-RAFT: write the address.
    dev.write_bscan(USER_LUT_ADDRESS, address)


# Write the word value into USER4.
def write_word(dev, value: int) -> None:
    # SK-RAFT: write the data word.
    dev.write_bscan(USER_LUT_DATA, value)


# Read back the word from USER4.
def read_word(dev) -> int:
    # SK-RAFT: read the data word.
    data = dev.read_bscan(USER_LUT_DATA)
    return int.from_bytes(data, "big") & LUT_DATA_MASK

### 7: Connect to the server

In [6]:
# A background thread monitors the Enter key. Thus, the injection loop
# below does not wait for input. Press Enter at any time to stop the
# current step early.
client = SkraftFpga.connect(ip=SK_RAFT_SERVER_IP, port=SK_RAFT_SERVER_PORT)
dev = client.attach(SERIAL_NUMBER)

### 8: Program the board...
...and upload the design to the server because it needs it for some operations.

In [7]:
dev.uploadDesignToServer(design)
dev.programFPGA(design)
print(f"programmed {SERIAL_NUMBER} with {design.bit}")

programmed 210183BA2524A with ../hw/design_examples/basys3/ff_lut_top/ff_lut_top.bit


### 9: Run the experiments!
The `N_OF_INJECTIONS` is the number of fault injections the experiment will do.

In [8]:
N_OF_INJECTIONS = 100


# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

# Part 1: flip-flop shift register.
# Make sure that the selected pblock contains flip-flops.
bits = dev.bit_count(TargetMemory.FF, pblock=PBLOCK_SR)
print(f"Part 1: {bits} flip-flop bits inside {PBLOCK_SR}.")
if bits == 0:
    print("Nothing to inject in Part 1: no flip-flop of "
          "the LL file falls inside the pblock. Check "
          "that the XDC, the LL and the bitstream "
          "belong together.")
    raise SystemExit("Stopping here")
print("     #  FAR       WORD  BIT  BEFORE    AFTER")
enable_clock(dev, True)
iterations = N_OF_INJECTIONS
n = 0
while iterations!=0:
    time.sleep(2)
    # Injection guideline (application memory): each upset
    # is a readback-modify-writeback of full frames. These
    # frames also hold adjacent logic. The server does
    # not protect against user activity. Thus, stop the
    # clock during the injection. This keeps the target idle.
    # Then start the clock again.
    enable_clock(dev, False)
    before = read_sr(dev)
    
    # SK-RAFT: flip one flip-flop bit.
    report = dev.inject_random(TargetMemory.FF, pblock=PBLOCK_SR)

    after = read_sr(dev)
    # FAR/word/bit is the location of the flip-flop in the
    # LL file. This location is the bit of the net that the
    # flip-flop drives. This bit is the value that the
    # flip-flop holds.
    # This script does not show how the server flips the bit.
    # BEFORE/AFTER show which of the 8 flip-flops changed.
    # The flip-flop holds the flipped value. When the clock
    # starts again, the value moves along the feedback
    # loop. You can see the value on the LEDs and through
    # BSCAN.
    print(f"{n + 1:>6}  {report.far:08x}  "
          f"{report.word_index:>4}  {report.bit_index:>3}  "
          f"{before:08b}  {after:08b}")
    n += 1
    iterations -= 1
    enable_clock(dev, True)


print(f"Part 1: done after {n} trials.")

Part 1: 8 flip-flop bits inside pblock_ff_sr_inst.
     #  FAR       WORD  BIT  BEFORE    AFTER
     1  00400f9f    97   28  00000100  00100100
     2  00400f9f    97   28  10010000  10110000
     3  00400f9f    97    4  11000010  11000011
     4  00400f9f    97   28  00001111  00101111
     5  00400f9f    97   29  10111100  10111110
     6  00400f9f    97    3  11111010  11101010
     7  00400f9f    98   27  10101011  10100011
     8  00400f9f    98    2  10001110  10001010
     9  00400f9f    97    3  00101010  00111010
    10  00400f9f    97    3  11101000  11111000
    11  00400f9f    97    3  11100011  11110011
    12  00400f9f    97   29  11001111  11001101
    13  00400f9f    97   29  00110111  00110101
    14  00400f9f    98    1  11010100  10010100
    15  00400f9f    97    3  01010010  01000010
    16  00400f9f    98   26  00001001  10001001
    17  00400f9f    98   27  00100110  00101110
    18  00400f9f    97   28  10111000  10011000
    19  00400f9f    97   28  01100010  0

In [11]:
print("Re-programming before Part 2.")
dev.programFPGA(design)

Re-programming before Part 2.


In [12]:
N_OF_INJECTIONS = 100

# Setup the SEED for deterministic behaviour. 
dev.configure_seed(0)

# Part 2: LUTRAM.
# When the script programs the FPGA again, USER1 resets to
# 0. The LUTRAM writes operate only when the gated clock
# runs. Thus, start the clock before the fill.
enable_clock(dev, True)
# Fill the memory array with a known pattern.
for address in range(LUT_DEPTH):
    # select() and write_word() are separate TAP scans.
    # The second scan cannot start before the first scan is
    # done. Thus, no delay is necessary between them.
    select(dev, address)
    write_word(dev, (0xA5A5A5A5 ^ address) & LUT_DATA_MASK)
# Read back each word and make sure that the write was
# successful.
for address in range(LUT_DEPTH):
    select(dev, address)
    word = read_word(dev)
    expected_word = (0xA5A5A5A5 ^ address) & LUT_DATA_MASK
    if word != expected_word:
        print(f"Part 2: address {address} reads "
              f"{word:#010x}, expected {expected_word:#010x}; "
              f"not injecting anything. Check USER3/USER4 "
              f"register ids and widths.")
        raise SystemExit("Stopping here")
                

print(f"Part 2: all {LUT_DEPTH} LUTRAM words read back "
      f"correctly. Injecting random upsets, not "
      f"corrected")
bits = dev.bit_count(TargetMemory.LUTRAM, pblock=PBLOCK_LUTRAM)
print(f"Part 2: {bits} LUTRAM bits inside "
      f"{PBLOCK_LUTRAM}.")
# Stop the clock before the injection (USER1 also gates the
# LUTRAM). Refer to the guideline in Part 1.
enable_clock(dev, False)
print("     #  FAR       WORD  BIT")
# Inject upsets until you press Enter.
iterations = N_OF_INJECTIONS
n = 0
while iterations!=0:
    report = dev.inject_random(TargetMemory.LUTRAM, pblock=PBLOCK_LUTRAM)
    print(f"{n + 1:>6}  {report.far:08x}  {report.word_index:>4}  {report.bit_index:>3}")
    # Wait 0.5 s to control the number of injections.
    time.sleep(0.5)
    n += 1
    iterations -= 1

print(f"Part 2: {n} upsets injected.")
# Start the clock to read the contents of the LUTRAM.
enable_clock(dev, True)
# Print the bits of each LUTRAM word, the value that the
# script wrote to the word, and the corrupted bits.
corrupt_words = 0
corrupt_bits = 0
print(f"{'ADDR':>6}  {'CORRECT':<{LUT_DATA_LEN}} {'READ':<{LUT_DATA_LEN}}  CORRUPTED BITS")
for address in range(LUT_DEPTH):
    select(dev, address)
    word = read_word(dev)
    expected_word = (0xA5A5A5A5 ^ address) & LUT_DATA_MASK
    # The bit positions (0 = LSB) that changed from the
    # written value.
    flipped = [bit for bit in range(LUT_DATA_LEN) if (word ^ expected_word) >> bit & 1]
    print(f"{address:>6}  {expected_word:0{LUT_DATA_LEN}b}  "
    f"{word:0{LUT_DATA_LEN}b}  "
    f"{', '.join(str(bit) for bit in flipped) or '-'}")
    if flipped:
        corrupt_words += 1
        corrupt_bits += len(flipped)
print(f"Part 2: {corrupt_words}/{LUT_DEPTH} words are "
f"corrupted, {corrupt_bits} bits in total.")

Part 2: all 64 LUTRAM words read back correctly. Injecting random upsets, not corrected
Part 2: 2048 LUTRAM bits inside pblock_LUTRAM.
     #  FAR       WORD  BIT
     1  00400a23   100   26
     2  00400a21    93   18
     3  00400a23   100   25
     4  00400a22    90   10
     5  00000a21     3   27
     6  00000a23     3    0
     7  00400a21    93   15
     8  00400a21    97    5
     9  00400a22    89    8
    10  00400a23    91   16
    11  00400a23   100    8
    12  00400a22    98   27
    13  00000a23     1   11
    14  00400a20    93   28
    15  00400a23    92   24
    16  00400a21    99   14
    17  00400a23    90   31
    18  00400a23    95    2
    19  00400a23    92    2
    20  00000a20     2   30
    21  00400a21    89   11
    22  00400a22    97   20
    23  00400a23    92   10
    24  00000a21     3   28
    25  00000a21     0    1
    26  00400a21   100   10
    27  00400a21    92    0
    28  00400a22    95   16
    29  00400a20    89   15
    30  00000a20     1   

In [13]:
print("Re-programming to clear the device.")
dev.programFPGA(design)

Re-programming to clear the device.
